In [112]:
from dataclasses import dataclass
from contextlib import contextmanager
import portion as P
import interval_logic as il

@dataclass
class Contract:
    assumption: il.Expression
    guarantee: il.Expression

    def __str__(self) -> str:
        return f"\u27E8 {self.assumption} \u2223 {self.guarantee} \u27E9"

    def merge(self, other: 'Contract') -> 'Contract':
        """Merge two contracts by combining their assumptions and guarantees."""
        return Contract(il.And((self.assumption, other.assumption)),
                        il.Or((il.And((self.guarantee, other.guarantee)),
                              il.Not(il.And((self.assumption, other.assumption))))))

    def separate(self, other: 'Contract') -> 'Contract':
        """Separate the contract into its assumption and guarantee."""
        return Contract(il.Or((il.And((self.assumption, other.guarantee)),
                               il.Not(il.And((self.guarantee, other.assumption))))),
                        il.And((self.guarantee, other.assumption)))
    
    def simplify(self) -> 'Contract':
        """Simplify the contract by removing redundant assumptions and guarantees."""
        return Contract(self.assumption.simplify(), self.guarantee.simplify())

    def replace(self, fn) -> 'Contract':
        """Replace parts of the contract using a given function."""
        return Contract(self.assumption.replace(fn), self.guarantee.replace(fn))

    def is_consistent(self, universe: P.Interval) -> bool:
        """Check if the contract is consistent (i.e., the assumption implies the guarantee)."""
        if self.guarantee == il.Bool(True):
            return True
        return il.denote(il.And((self.assumption, self.guarantee)),
                         {}, universe) == universe

@contextmanager
def step(msg: str, disable: int = 4):

    counter = list(getattr(step, "counter", (0,)))
    counter[-1] += 1
    if len(counter) == 1:
        println(f"Step {counter[0]}: {msg}")
    else:
        println(f"Step {'.'.join(map(str, counter))}: {msg}", marker=["├ ", "│ "])
    step.counter = tuple(counter + [0])
    _disable = getattr(step, "disable", 4)
    step.disable = disable
    try:
        yield
    finally:
        pass
    step.disable = _disable
    step.counter = step.counter[:-1]
    if len(step.counter) == 1:
        println('╯')
    else:
        println('', marker=["├─╯ ", "│ "])

def println(msg: str = "", *, marker: str | list[str] = "│ "):
    indent = len(getattr(step, "counter", (0,))) - 1
    if indent >= getattr(step, "disable", 4):
        return
    ms = [marker] if isinstance(marker, str) else list(marker)
    ms += [ms[-1]] * (indent - len(ms)) # may be more than `indent` markers
    ws = "".join(reversed(ms[:indent]))
    print(ws + str(msg))

def printeq(*args):
    if len(args) == 0: return
    println(args[0])
    for arg in args[1:]:
        println(f"= {arg}")

In [ ]:
step.counter = (0,)
UNIVERSE = P.closed(0, 5)

#??
signals = {
    'j': il.NaturalSignal({
        11: P.closed(1, 3),
    })
}

with step('Initial Contract'):
    printeq(C_cmt := Contract(il.Bool(True), il.Bool(True)))

with step('Request 1'):

    with step('Incoming Request'):
        printeq(C_req1 := Contract(il.Exists(P.from_string('[1, 2)', float), il.Signal('j').equals(11)), il.Bool(True)))
        # This type of contract is definitionally inconsistent, but 
        # with the implication-interpretation it seems correct:
        #       ⊤ -> ∃[1.0,2.0): j = 1
        #     = ∃[1.0,2.0): j = 1
        # But is it important for a request to be consistent? Perhaps not... 

    with step('Candidate Contract'):
        printeq(C_cand1 := C_cmt.merge(C_req1),
                C_cand1 := C_cand1.simplify())

    with step('Create Offer Contract'):
        if C_cand1.is_consistent(UNIVERSE):
            with step('Candidate Contract is Consistent'):
                phi = C_cand1.assumption
                if isinstance(phi, il.Exists):
                    phi = il.ForAll(phi.interval, phi.body)
                printeq(C_ofr1 := Contract(il.Bool(True), phi))

    with step('Merge Accepted Offer'):
        printeq(C_cmt := C_cmt.merge(C_ofr1),
                C_cmt := C_cmt.simplify())
    
with step('Request 2'):
        
    with step('Incoming Request'):
        printeq(C_req2 := Contract(il.Exists(P.from_string('[2, 4)', float), il.Signal('j').equals(22)), il.Bool(True)))

    with step('Candidate Contract'):
        printeq(C_cand2 := C_cmt.merge(C_req2),
                C_cand2 := C_cand2.simplify())

    

Step 1: Initial Contract
│ \ ⊤ | ⊤ \
╯
Step 2: Request 1
├ Step 2.1: Incoming Request
│ │ \ ∃[1.0,2.0): j = 11 | ⊤ \
├─╯ 
├ Step 2.2: Candidate Contract
│ │ \ ⊤ ∧ (∃[1.0,2.0): j = 11) | ⊤ ∧ ⊤ ∨ ¬(⊤ ∧ (∃[1.0,2.0): j = 11)) \
│ │ = \ ∃[1.0,2.0): j = 11 | ⊤ \
├─╯ 
├ Step 2.3: Create Offer Contract
│ ├ Step 2.3.1: Candidate Contract is Consistent
│ │ │ \ ⊤ | ∀[1.0,2.0): j = 11 \
│ ├─╯ 
├─╯ 
├ Step 2.4: Merge Accepted Offer
│ │ \ ⊤ ∧ ⊤ | ⊤ ∧ (∀[1.0,2.0): j = 11) ∨ ¬(⊤ ∧ ⊤) \
│ │ = \ ⊤ | ∀[1.0,2.0): j = 11 \
├─╯ 
╯
Step 3: Request 2
├ Step 3.1: Incoming Request
│ │ \ ∃[2.0,4.0): j = 22 | ⊤ \
├─╯ 
├ Step 3.2: Candidate Contract
│ │ \ ⊤ ∧ (∃[2.0,4.0): j = 22) | (∀[1.0,2.0): j = 11) ∧ ⊤ ∨ ¬(⊤ ∧ (∃[2.0,4.0): j = 22)) \
│ │ = \ ∃[2.0,4.0): j = 22 | ¬(∃[2.0,4.0): j = 22) ∨ (∀[1.0,2.0): j = 11) \
├─╯ 
╯


In [115]:
C_ofr1.guarantee

ForAll(interval=[1.0,2.0), body=Equal(left=Signal(name='j'), right=Natural(value=11)))

$$
\begin{split}
C^\text{cand}_1 =& \langle \top \mid \exists [1, 2): j = 1 \rangle
\end{split}
$$

In [ ]:
with step('Conflicting universal guarantees'):
    printeq(C_left := Contract(il.TOP, il.ForAll('[1, 3)', il.Signal('j').equals(1))))

    printeq(C_right := Contract(il.TOP, il.ForAll('[2, 4)', il.Signal('j').equals(2))))

    printeq(C_overlap := C_left.merge(C_right),
            C_overlap := C_overlap.simplify())

In [ ]:
def postproc(phi, n):

    def keep(n):
        j = il.Signal('j')
        n = il.Natural(n)
        def replacer(psi):
            match psi:
                case il.Equal(left, right):
                    if left == j and right != n:
                        return il.Not(j.equals(n))
                    elif right == j and left != n:
                        return il.Not(j.equals(n))
            return psi
        return replacer

    def pos(psi):
        match psi:
            case il.And((left, right)): pass
            case _: return psi
        match (left, right):
            case (il.ForAll(I, p), il.Exists(J, q)):
                if p == il.Not(q):
                    return il.Exists(J - I, q)
            case (il.Exists(J, q), il.ForAll(I, p)):
                if p == il.Not(q):
                    return il.Exists(J - I, q)
        return psi

    def flip(psi):
        match psi:
            case il.Exists(I, p):
                return il.ForAll(I, p)
        return psi

    return phi.replace(keep(n)).replace(pos).replace(flip)

printeq(phi := C_overlap.guarantee.simplify(),
        phi := postproc(phi, 2).simplify())